In [ ]:
import pandas as pd

In [ ]:
#Load Data
paired = pd.read_csv("/mloscratch/users/clerget/NeuroMeditron/src_GAMMA/paired_healthcode.csv")
train_split = pd.read_csv("/mloscratch/users/clerget/data/data_paired/10_fold_CV/processed_paired/paired_splits/balanced_train/healthcode_10fold_train.csv")
valtest_split = pd.read_csv("/mloscratch/users/clerget/data/data_paired/10_fold_CV/processed_paired/paired_splits/balanced_train/healthcode_10fold_val_test.csv")

In [5]:
print ("paired dataset shape:", paired.shape)
paired.head()

paired dataset shape: (532, 3)


,healthCode,label_PD,modality
0,00081bd9-9abd-4003-b035-de6cc3e8c922,0,paired
1,0160664a-f4af-4071-a4aa-2967f3ea0503,1,paired
2,01aea090-d2cd-409e-8ce8-22887ada7af8,1,paired
3,01b1ec31-0348-4148-a641-626d4391b4fb,0,paired
4,01fa5bda-f11e-47ef-94c6-37697ad26a86,1,paired


In [6]:
print ("train split shape:", train_split.shape)
train_split.head()

train split shape: (3680, 3)


,fold_iteration,healthCode,subset
0,0,00081bd9-9abd-4003-b035-de6cc3e8c922,train
1,0,0160664a-f4af-4071-a4aa-2967f3ea0503,train
2,0,01aea090-d2cd-409e-8ce8-22887ada7af8,train
3,0,01b1ec31-0348-4148-a641-626d4391b4fb,train
4,0,0309d5a8-79cd-460a-9a87-6902460ab71d,train


In [7]:
print ("val/test split shape:", valtest_split.shape)
valtest_split.head()

val/test split shape: (1064, 3)


,fold_iteration,healthCode,subset
0,0,01fa5bda-f11e-47ef-94c6-37697ad26a86,val
1,0,030fe814-de84-4fb5-8df6-84dd6e7cc7dc,val
2,0,05d4e71a-c792-4549-8ec7-6a7dc355a1bd,val
3,0,09670b1f-23a3-46a5-8daf-f78f577b7697,val
4,0,13627002-073b-4366-8889-9561789c51be,val


In [8]:
#Compare healthCodes across datasets
demo_codes = set(paired["healthCode"].unique())
train_codes = set(train_split["healthCode"].unique())
valtest_codes = set(valtest_split["healthCode"].unique())

print(f"Paired dataset: {len(demo_codes)} unique healthCodes")
print(f"Train split: {len(train_codes)} unique healthCodes")
print(f"Val/Test split: {len(valtest_codes)} unique healthCodes")

Paired dataset: 532 unique healthCodes
Train split: 532 unique healthCodes
Val/Test split: 532 unique healthCodes


**Tapping Features Extraction**

In [ ]:
tapping_sequences = pd.read_csv("/mloscratch/users/clerget/NeuroMeditron/src_GAMMA/tapping_model/csv/tapping_event_sequences.csv")
print("Tapping event sequences shape:", tapping_sequences.shape)
print(tapping_sequences.head())

Tapping event sequences shape: (3498391, 3)
                             healthCode   delta_t             button
0  5134777c-3563-41a5-abcf-cce272a2ebf3  0.429935  TappedButtonRight
1  5134777c-3563-41a5-abcf-cce272a2ebf3  0.349959   TappedButtonLeft
2  5134777c-3563-41a5-abcf-cce272a2ebf3  0.283382  TappedButtonRight
3  5134777c-3563-41a5-abcf-cce272a2ebf3  0.299909   TappedButtonLeft
4  5134777c-3563-41a5-abcf-cce272a2ebf3  0.300355   TappedButtonLeft


In [ ]:
tapping_features = pd.read_csv("/mloscratch/users/clerget/NeuroMeditron/src_GAMMA/tapping_model/csv/tapping_statistical_features.csv")
print("\nTapping statistical features shape:", tapping_features.shape)
print(tapping_features.head())


Tapping statistical features shape: (15992, 12)
    mean_dt    std_dt  lr_ratio  left_count  right_count  total_taps  \
0  0.131566  0.115565  0.853659          70           82         152   
1  0.080679  0.081303  0.871212         115          132         247   
2  0.181918  0.114110  1.000000          54           54         108   
3  0.197994  0.122458  0.960000          48           50         101   
4  0.084800  0.071506  1.201923         125          104         236   

       mean_x      mean_y      std_x      std_y  missed_taps  \
0  143.362745  434.088235  73.065760  16.088107            0   
1  151.334677  400.203629  60.303491  14.125084            0   
2  153.674312  405.220183  78.037640  20.139813            0   
3  153.299020  437.220588  77.773624  22.610467            3   
4  178.966245  427.729958  67.726390  15.981294            7   

                             healthCode  
0  5134777c-3563-41a5-abcf-cce272a2ebf3  
1  3516da76-a3e1-47ac-9255-ee100797ba9b  
2  0a46

In [15]:
tapping_with_diagnosis = tapping_features.merge(
    paired[["healthCode", "label_PD"]], 
    on="healthCode", 
    how="left"
)

print("Tapping features merged with diagnosis:")
print(tapping_with_diagnosis.head())

Tapping features merged with diagnosis:
    mean_dt    std_dt  lr_ratio  left_count  right_count  total_taps  \
0  0.131566  0.115565  0.853659          70           82         152   
1  0.080679  0.081303  0.871212         115          132         247   
2  0.181918  0.114110  1.000000          54           54         108   
3  0.197994  0.122458  0.960000          48           50         101   
4  0.084800  0.071506  1.201923         125          104         236   

       mean_x      mean_y      std_x      std_y  missed_taps  \
0  143.362745  434.088235  73.065760  16.088107            0   
1  151.334677  400.203629  60.303491  14.125084            0   
2  153.674312  405.220183  78.037640  20.139813            0   
3  153.299020  437.220588  77.773624  22.610467            3   
4  178.966245  427.729958  67.726390  15.981294            7   

                             healthCode  label_PD  
0  5134777c-3563-41a5-abcf-cce272a2ebf3       1.0  
1  3516da76-a3e1-47ac-9255-ee100797ba9

In [14]:
print("Mean Δt by diagnosis:")
print(tapping_with_diagnosis.groupby("label_PD")["mean_dt"].describe())

print("\nL/R ratio by diagnosis:")
print(tapping_with_diagnosis.groupby("label_PD")["lr_ratio"].describe())

Mean Δt by diagnosis:
            count      mean       std       min       25%       50%       75%  \
label_PD                                                                        
0.0        3378.0  0.094790  0.048748  0.036418  0.057239  0.079994  0.118239   
1.0       12387.0  0.125359  0.078341  0.037261  0.076665  0.105511  0.149491   

               max  
label_PD            
0.0       0.455010  
1.0       0.929778  

L/R ratio by diagnosis:
            count      mean       std       min       25%       50%       75%  \
label_PD                                                                        
0.0        3378.0  1.018129  0.162252  0.171053  0.976744  1.000000  1.039152   
1.0       12387.0  1.105898  1.182022  0.000000  0.951807  1.009804  1.110000   

                max  
label_PD             
0.0        5.133333  
1.0       90.000000  


Count is the number of records (trials) per diagnosis class. 
- Diagnosis 0 (Healthy): 16,964 trial records
- Diagnosis 1 (PD): 38,953 trial records

50% is the median
For diagnosis 0 (Healthy): 
- 25% = 0.052575 → 25% of healthy trials have Δt ≤ 0.052575 seconds
- 50% = 0.057465 → 50% have Δt ≤ 0.057465 (this is the median)
- 75% = 0.103295 → 75% have Δt ≤ 0.103295 seconds

**ADVANCED FEATURES** one row per patient

**Fatigue slope**: 
- average reaction time in the 1st quarter 
- average reaction time in the 4th quarter
- fatigue slope = (last-first)/first = percentage increase/decrease (if negative)

**Variability index**
- dt mean 
- std mean 
- variability index = std/mean = coefficient of variation 

**Tap distance**
- euclidian distance between consecutive taps 
- mean distance 
- std distance 

**Fluctuation**
- dt_diff changes in reaction tieme from tap to tap 
- std (dt_diff) if high = unpredictable tapping pattern 

In [ ]:
advanced_features_patient = pd.read_csv("/mloscratch/users/clerget/NeuroMeditron/src_GAMMA/tapping_model/csv/tapping_advanced_features_patient.csv")
print("\nAdvanced tapping features shape:", advanced_features_patient.shape)
print(advanced_features_patient.head())


Advanced tapping features shape: (608, 11)
                             healthCode  fatigue_slope_mean  \
0  00081bd9-9abd-4003-b035-de6cc3e8c922            0.312848   
1  005908f8-2bf2-4855-ab2d-ac3c0190b8fb            0.035746   
2  00d9a01f-08ae-4c3d-9b23-919f66fb066f            0.092835   
3  0160664a-f4af-4071-a4aa-2967f3ea0503            0.129422   
4  01aea090-d2cd-409e-8ce8-22887ada7af8           -0.122193   

   fatigue_slope_std  variability_index_mean  variability_index_std  \
0           0.320994                0.350326               0.190731   
1                NaN                0.781290                    NaN   
2           0.113944                1.203761               0.183419   
3           0.137016                0.848255               0.121591   
4           0.195725                0.562091               0.150828   

   mean_tap_distance_mean  mean_tap_distance_std  std_tap_distance_mean  \
0              149.108105              13.185257              37.272614   


**NaN** because if one trial, cannot compute std

In [24]:
# Calculate mean of fatigue_slope_mean across all patients
mean_fatigue_slope = advanced_features_patient["fatigue_slope_mean"].mean()

print(f"Mean fatigue_slope_mean across all patients: {mean_fatigue_slope:.6f}")

# Also show by diagnosis
print("\nMean fatigue_slope_mean by diagnosis:")
merged = advanced_features_patient.merge(
    paired[["healthCode", "label_PD"]], 
    on="healthCode", 
    how="left"
)
print(merged.groupby("label_PD")["fatigue_slope_mean"].describe())

Mean fatigue_slope_mean across all patients: 0.051810

Mean fatigue_slope_mean by diagnosis:
          count      mean       std       min       25%       50%       75%  \
label_PD                                                                      
0.0       230.0  0.066231  0.157428 -0.413070 -0.030536  0.074020  0.153172   
1.0       302.0  0.043469  0.209001 -0.772777 -0.065834  0.032554  0.133039   

               max  
label_PD            
0.0       0.575797  
1.0       1.067536  


**ADVANCED FEATURES** many rows per subject

In [ ]:
advanced_features_session = pd.read_csv("/mloscratch/users/clerget/NeuroMeditron/src_GAMMA/tapping_model/csv/tapping_advanced_features_session.csv")
print("\nAdvanced tapping features shape:", advanced_features_session.shape)
print(advanced_features_session.head())


Advanced tapping features shape: (15992, 6)
   fatigue_slope  variability_index  mean_tap_distance  std_tap_distance  \
0      -0.209819           0.878379         113.285935         61.770407   
1       0.085573           1.007742         106.095709         39.966436   
2       0.074332           0.627261         122.535063         64.082528   
3       0.040705           0.618496         135.062383         52.932558   
4       0.043829           0.843230         119.798322         43.549124   

   fluctuation                            healthCode  
0     0.185746  5134777c-3563-41a5-abcf-cce272a2ebf3  
1     0.142372  3516da76-a3e1-47ac-9255-ee100797ba9b  
2     0.201833  0a4662b0-9ee3-4029-af85-7e342cb87810  
3     0.215947  e787d06a-bcce-4ef0-848e-8b319fc7ce99  
4     0.126878  3516da76-a3e1-47ac-9255-ee100797ba9b  
